# EPUB Audiobook - remote chunk synthesis

This notebook synthesizes the text chunks exported by the EPUB Audiobook App
for **one patch**, using the same `VoxCPM2` model the app uses locally, and
writes the resulting `chunk_NNN.wav` files into an **`output/`** subfolder inside
the exported folder, so the app's **Import from Drive** button can pick them up.

> **Enable a GPU first.** This model runs on CUDA. In Colab: **Runtime > Change runtime type > GPU (T4)**, then restart the session. Pick **GPU, not TPU** - VoxCPM cannot use a TPU and will silently fall back to CPU (extremely slow). Cell 5 checks this for you.


It reads everything it needs from `manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## Google Colab (recommended)
The app already uploaded this folder into **your own Google Drive** (the
account you connected). Just run the cells top to bottom: cell 2 mounts your
Drive, and the folder is a normal filesystem path from then on - no Google API
calls needed in this notebook at all.

## Kaggle
Kaggle has no native Google Drive mount. The practical flow there:
1. In the app, use **Download package locally** (instead of, or in addition
   to, exporting to Drive) to get a `.zip` of this same folder.
2. Upload that zip as a Kaggle Dataset and attach it to this notebook.
3. Skip the Drive-mount cell below, and instead set `FOLDER_PATH` to your
   Kaggle dataset input path (e.g. `/kaggle/input/<dataset-name>`).
4. After running, download the generated `chunk_NNN.wav` files from Kaggle's
   output pane and drop them into an `output/` subfolder inside the matching
   Drive folder yourself (drag & drop in the Drive web UI, or re-upload), or
   upload them (or a .zip of them) directly in the app's chunk page instead -
   either way works, the app looks for `output/` first and falls back to the
   folder's top level for older exports.

In [ ]:
# Cell 1: install dependencies
!pip install -q voxcpm soundfile numpy

In [ ]:
# Cell 2: Google Colab only - mount your Drive. The exported folder is located
# automatically by patch id, so you do NOT need to paste the folder name by hand.
# Skip this cell entirely on Kaggle (use Cell 3 instead).
from google.colab import drive
import glob, json, os

drive.mount('/content/drive')

PATCH_ID = __PATCH_ID__  # injected by the app when this notebook was exported
EXPORTS_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports"
DEFAULT_FOLDER = os.path.join(EXPORTS_ROOT, "__DEFAULT_FOLDER_NAME__")

# Scan every export folder and match the one whose manifest.json has our patch id.
FOLDER_PATH = None
for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*")), reverse=True):
    manifest_path = os.path.join(d, "manifest.json")
    if not os.path.isfile(manifest_path):
        continue
    try:
        with open(manifest_path, encoding="utf-8") as f:
            if json.load(f).get("patch_id") == PATCH_ID:
                FOLDER_PATH = d
                break
    except Exception:
        pass

if FOLDER_PATH is None:
    FOLDER_PATH = DEFAULT_FOLDER  # fall back to the exact name the app used at export time

print("Using folder:", FOLDER_PATH)
assert os.path.isdir(FOLDER_PATH), (
    f"Folder not found: {FOLDER_PATH}\n"
    "Make sure the export was uploaded to this Google account's Drive, "
    "or set FOLDER_PATH manually."
)

In [ ]:
# Cell 3: Kaggle only - point at the attached dataset instead of Drive.
# FOLDER_PATH = "/kaggle/input/<dataset-name>"

In [ ]:
# Cell 4: load the manifest and the voice reference clip (if the book uses voice cloning)
import json
import os

with open(os.path.join(FOLDER_PATH, "manifest.json"), "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Patch {manifest['patch_id']} of book '{manifest['book_title']}'")
print(f"{manifest['chunk_count']} chunks, max_chars={manifest['max_chars']}")

reference_wav_path = None
prompt_text = None
if manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, manifest["reference_wav"])
    prompt_text = manifest.get("reference_transcript") or None
    print(f"Using cloned voice reference: {reference_wav_path}")

In [ ]:
# Cell 5: check you're actually on a GPU (VoxCPM is far too slow on CPU).
# If this stops with an error, go to Runtime > Change runtime type > Hardware
# accelerator > GPU (T4), then Runtime > Restart session and run again.
# NOTE: pick GPU, NOT TPU - VoxCPM uses CUDA and cannot run on a TPU.
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected - VoxCPM would run on CPU and be extremely slow. "
        "Colab: Runtime > Change runtime type > GPU (T4), then Restart session. "
        "Choose GPU, not TPU. On Kaggle: enable a GPU accelerator in the sidebar."
    )

In [ ]:
# Cell 6: load the model (same model id the app uses locally)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained(manifest.get("voxcpm_model_id", "openbmb/VoxCPM2"), load_denoiser=False)

In [ ]:
# Cell 7: synthesize the chunks and write chunk_NNN.wav into an 'output' subfolder
# (keeps generated audio separate from the exported chunk_NNN.txt/manifest files).
import soundfile as sf

# --- config: which chunks to run in THIS session ---
# Use these to split work across accounts/runs or to skip early chunks.
START_INDEX = 0      # first chunk to synthesize (e.g. 1 to start at chunk_001)
END_INDEX = None     # last chunk to synthesize, inclusive; None = go to the end
SKIP_EXISTING = True # skip chunks that already have a .wav (safe to resume)

OUTPUT_DIR = os.path.join(FOLDER_PATH, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

sample_rate = model.tts_model.sample_rate

for chunk_filename in manifest["chunks"]:
    index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
    n = int(index)
    if n < START_INDEX:
        continue
    if END_INDEX is not None and n > END_INDEX:
        break

    out_path = os.path.join(OUTPUT_DIR, f"chunk_{index}.wav")
    if SKIP_EXISTING and os.path.exists(out_path):
        print(f"skip {chunk_filename} (already synthesized)")
        continue

    with open(os.path.join(FOLDER_PATH, chunk_filename), "r", encoding="utf-8") as f:
        text = f.read()

    kwargs = {}
    if reference_wav_path:
        kwargs["reference_wav_path"] = reference_wav_path
        if prompt_text:
            kwargs["prompt_wav_path"] = reference_wav_path
            kwargs["prompt_text"] = prompt_text

    audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
    sf.write(out_path, audio, sample_rate)
    print(f"wrote {out_path}")

print("Done. Go back to the app and import the results.")